# 06 - Demo E2E real con LangGraph

Este notebook muestra la motion completa: diagnostico S1/S2, scoring, imputacion de contacto, grafo LangGraph, envio real de email, listener de reply, clasificacion LLM, WhatsApp y handoff.

## Tabla de contenido

1. Configuracion y datos base
2. Por que S1 -> S2
3. Score y seleccion del lead
4. Imputacion de email y WhatsApp
5. Grafo LangGraph visible
6. E2E real paso a paso
7. Memoria local del agente en data/

## 1. Configuracion y datos base

El ranking oficial se recalcula desde el dataset y se compara conceptualmente con `analysis/top50.csv`. La base SQLite no se usa para crear el ranking; solo guarda interacciones del agente.

In [ ]:
from pathlib import Path
import os
import sys
import time
from datetime import datetime

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Carga simple de .env sin imprimir secretos.
env_path = ROOT / '.env'
if env_path.exists():
    for line in env_path.read_text(encoding='utf-8-sig').splitlines():
        if line.strip() and not line.strip().startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ.setdefault(key.strip(), value.strip())

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from src.scoring.compute_score import calcular_score
from src.db import repository
from src.agents.graph import compiled_graph, compiled_reply_graph
from src.outreach.email_service import send_email_d0
from live_demo.email_listener import wait_for_reply_and_classify

DATASET = ROOT / 'data' / 'GTM-Engineer-BC-Dataset.xlsx'
TOP50_CSV = ROOT / 'analysis' / 'top50.csv'

universo = pd.read_excel(DATASET, sheet_name='universo_potencial')
funnel = pd.read_excel(DATASET, sheet_name='funnel_snapshot')
top50_csv = pd.read_csv(TOP50_CSV)
top50_calculado = calcular_score(DATASET)

print(f'Universo base: {len(universo):,} marcas')
print(f'Funnel snapshot: {len(funnel):,} leads')
display(top50_calculado.head(10))

## 2. Por que S1 -> S2

Pregunta: donde esta el cuello operativo mas accionable para una motion automatizada?  
Metodo: SQL con DuckDB sobre `funnel_snapshot`, grafico y conclusion.

In [ ]:
con = duckdb.connect()
con.register('funnel_snapshot', funnel)

sql_stage = '''
SELECT
  stage,
  COUNT(*) AS n_leads,
  ROUND(AVG(days_since_last_touch), 1) AS avg_days_since_last_touch,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_pipeline
FROM funnel_snapshot
GROUP BY stage
ORDER BY
  CASE stage
    WHEN 'S1_Prospect' THEN 1 WHEN 'S2_Contacted' THEN 2 WHEN 'S3_Qualified' THEN 3
    WHEN 'S4_Demo_done' THEN 4 WHEN 'S5_Negotiation' THEN 5 WHEN 'S6_Contract_signed' THEN 6
    WHEN 'S7_Onboarding' THEN 7 WHEN 'S8_Live' THEN 8 ELSE 99 END
'''
stage_touch = con.sql(sql_stage).df()
display(stage_touch)

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(stage_touch['stage'], stage_touch['avg_days_since_last_touch'], color='#246B73')
ax.set_title('Dias sin toque promedio por stage')
ax.set_xlabel('Stage')
ax.set_ylabel('Dias sin toque promedio')
ax.tick_params(axis='x', rotation=35)
ax.bar_label(bars, fmt='%.1f', padding=3)
plt.tight_layout()
plt.show()

s1 = stage_touch.loc[stage_touch['stage'].eq('S1_Prospect')].iloc[0]
s2 = stage_touch.loc[stage_touch['stage'].eq('S2_Contacted')].iloc[0]
print(f'S1 concentra {s1.pct_pipeline:.1f}% del pipeline ({int(s1.n_leads)} leads) y promedia {s1.avg_days_since_last_touch:.1f} dias sin toque.')
print(f'S2 tiene {int(s2.n_leads)} leads y {s2.avg_days_since_last_touch:.1f} dias sin toque promedio.')

In [ ]:
sql_source = '''
WITH base AS (
  SELECT
    source,
    COUNT(*) AS n,
    SUM(CASE WHEN stage IN ('S3_Qualified','S4_Demo_done','S5_Negotiation','S6_Contract_signed','S7_Onboarding','S8_Live') THEN 1 ELSE 0 END) AS s3_plus
  FROM funnel_snapshot
  WHERE source IN ('BNPL_xsell', 'Outbound_cold', 'Referral')
  GROUP BY source
)
SELECT source, n, s3_plus, ROUND(100.0 * s3_plus / n, 1) AS cvr_s3_plus_pct
FROM base
ORDER BY cvr_s3_plus_pct DESC
'''
source_cvr = con.sql(sql_source).df()
display(source_cvr)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(source_cvr['source'], source_cvr['cvr_s3_plus_pct'], color='#8F5B2E')
ax.set_title('CVR a S3+ por source')
ax.set_ylabel('% a S3+')
for bar, (_, row) in zip(bars, source_cvr.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, f"{row.cvr_s3_plus_pct:.1f}%\nn={int(row.n)}", ha='center')
plt.tight_layout()
plt.show()

display(Markdown('**Conclusion:** S1 -> S2 es el punto de mayor impacto operativo porque permite aumentar cadencia antes de que el lead llegue a calificacion, y BNPL x-sell funciona como direccion de priorizacion aunque la muestra sea pequena.'))

## 3. Score y seleccion del lead

La demo no esta amarrada a una marca especifica. Por defecto toma el mejor Tier B calculado; si quieres forzar otra marca, llena `DEMO_BRAND_ID` antes de ejecutar la celda.

In [ ]:
DEMO_BRAND_ID = os.environ.get('DEMO_BRAND_ID', '').strip()  # opcional: 'Brand_XXXX'

candidatos = top50_calculado[top50_calculado['tier'].eq('B')].sort_values('final_score', ascending=False)
display(candidatos[['rank', 'brand_id', 'tier', 'category', 'gmv_cop_millions_12m', 'momentum_score', 'final_score', 'why']].head(10))

if DEMO_BRAND_ID:
    lead_row = top50_calculado[top50_calculado['brand_id'].eq(DEMO_BRAND_ID)]
else:
    lead_row = candidatos.head(1)

if lead_row.empty:
    raise ValueError(f'No existe DEMO_BRAND_ID={DEMO_BRAND_ID} en el top50 calculado')

brand = lead_row.iloc[0].dropna().to_dict()
print('Lead elegido para demo:')
display(pd.DataFrame([brand]))

## 4. Imputacion de email y WhatsApp

Estos dos campos se imputan para la prueba real. Todo lo demas viene del score calculado.

In [ ]:
# Puedes cambiar estos valores en la celda o definirlos en .env.
contacto_email = os.environ.get('DEMO_EMAIL_DESTINO') or input('Correo destino para la prueba real: ').strip()
contacto_whatsapp = os.environ.get('DEMO_WHATSAPP_NUMBER') or input('WhatsApp destino con indicativo, ejemplo +573001112233: ').strip()

brand['contacto_email'] = contacto_email
brand['contacto_whatsapp'] = contacto_whatsapp

repository.upsert_lead(
    brand['brand_id'],
    category=brand.get('category'),
    gmv_cop_millions_12m=float(brand.get('gmv_cop_millions_12m', 0)),
    tier=brand.get('tier'),
    contacto_email=contacto_email,
    contacto_whatsapp=contacto_whatsapp,
)
repository.grant_opt_in(brand['brand_id'], 'whatsapp')
repository.save_agent_interaction(
    run_id=brand['brand_id'],
    source='notebook_06',
    event_type='lead_demo_preparado',
    brand_id=brand['brand_id'],
    content=f"Lead preparado con email={contacto_email} y whatsapp={contacto_whatsapp}",
    metadata=brand,
)
print('Lead guardado en data/agent_memory.sqlite3 con opt-in WhatsApp para la demo.')

## 5. Grafo LangGraph visible

Hay dos grafos visibles: la motion inicial de contacto y la motion de reply. Ambos muestran nombres de nodos para poder explicar el flujo en una demo.

In [ ]:
mermaid_inicial = compiled_graph.get_graph().draw_mermaid()
mermaid_reply = compiled_reply_graph.get_graph().draw_mermaid()

display(Markdown('### Grafo 1: motion inicial\n```mermaid\n' + mermaid_inicial + '\n```'))
display(Markdown('### Grafo 2: procesamiento de reply\n```mermaid\n' + mermaid_reply + '\n```'))

# Tambien imprimimos el Mermaid crudo por si el visor no renderiza diagramas.
print('--- Grafo inicial ---')
print(mermaid_inicial)
print('\n--- Grafo reply ---')
print(mermaid_reply)

## 6. E2E real paso a paso

Paso A: enviar email HTML real.  
Paso B: responder ese correo.  
Paso C: el listener lee el reply del thread, lo pasa por LangGraph y dispara WhatsApp/Slack segun la clasificacion.

Para evitar envios accidentales, cambia `EJECUTAR_E2E_REAL` a `True` solo cuando los contactos esten correctos.

In [ ]:
EJECUTAR_E2E_REAL = False

required = ['GROQ_API_KEY', 'SLACK_WEBHOOK_URL', 'TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_WHATSAPP_FROM']
missing = [key for key in required if not os.environ.get(key)]
if missing:
    print('Faltan variables para envio real:', missing)

if EJECUTAR_E2E_REAL:
    sent_after_epoch_ms = int(time.time() * 1000)
    email_result = send_email_d0(brand, brand['contacto_email'], reply_to=os.environ.get('DEMO_REPLY_TO_EMAIL'), dry_run=False)
    thread_id = email_result['threadId']
    repository.upsert_lead(brand['brand_id'], thread_id=thread_id)
    repository.mark_contacted(brand['brand_id'], thread_id=thread_id)
    repository.save_agent_interaction(
        run_id=brand['brand_id'], source='notebook_06', event_type='email_real_enviado',
        brand_id=brand['brand_id'], content=f"thread_id={thread_id}", metadata=email_result,
    )
    print('Email HTML enviado. Responde el correo; luego ejecuta la siguiente celda del listener.')
else:
    thread_id = None
    sent_after_epoch_ms = None
    print('Modo seguro: no se envio email. Cambia EJECUTAR_E2E_REAL=True para la prueba real.')

In [ ]:
if EJECUTAR_E2E_REAL and thread_id:
    state = wait_for_reply_and_classify(
        thread_id=thread_id,
        brand_data=brand,
        poll_seconds=15,
        after_epoch_ms=sent_after_epoch_ms,
        allow_sent_demo_fallback=True,
        process_with_langgraph=True,
        dry_run=False,
    )
    print('Decision:', state.get('decision'))
    print('Reply:', state.get('reply_recibido'))
    print('Clasificacion:', state.get('clasificacion'))
    print('WhatsApp:', state.get('whatsapp_result'))
    print('\nLog LangGraph:')
    print('\n'.join(state.get('log_razonamiento', [])))
else:
    print('Listener no ejecutado porque el envio real esta apagado o no hay thread_id.')

## 7. Memoria local del agente en data/

Cada paso relevante se guarda en `data/agent_memory.sqlite3`, tabla `agent_interactions`. Esto funciona como memoria auditable y busqueda local tipo RAG simple.

In [ ]:
memoria = repository.list_agent_interactions(limit=20)
display(pd.DataFrame(memoria))

print('Busqueda local por brand_id:')
display(pd.DataFrame(repository.search_agent_interactions(str(brand['brand_id']), limit=10)))